In [49]:
from openai import OpenAI
import json
import os
from crewai import Agent, Crew, Task, Process, LLM

from dotenv import load_dotenv

load_dotenv()  # loads from .env by default
os.environ["OPENAI_API_KEY"]=os.getenv("OPENAI_API_KEY")

In [50]:
# Define the LLM directly using openai.OpenAI
openai_client = OpenAI()

openai_llm = LLM(
    model="gpt-4-turbo-preview", # Specify the model name
    client=openai_client,      # Pass the OpenAI client instance
    temperature=0.7             # Optional parameters like temperature
)

# Agent without Tools

In [51]:
# Create Agents, passing the openai_llm instance
manager_agent = Agent(
    role="Project Manager",
    goal="Successfully manage and delegate tasks to achieve project goals. If you are invoking a tool for any task, please explicitly share the name of the tool also.",
    backstory="Experienced project manager with a strong track record of delivering complex projects on time and within budget.",
    llm=openai_llm,
    allow_delegation=True,
    verbose=True
)

researcher_agent = Agent(
    role="Market Researcher",
    goal="Conduct thorough market research and identify key trends.",
    backstory="Detail-oriented market researcher with expertise in data analysis and trend forecasting.",
    llm=openai_llm,
    verbose=True
)

writer_agent = Agent(
    role="Content Writer",
    goal="Create high-quality and engaging content based on research findings.",
    backstory="Creative and skilled content writer with a passion for clear and concise communication.",
    llm=openai_llm,
    verbose=True
)


In [52]:
# Create Tasks (no change needed)
research_task = Task(
    description="Conduct market research on the latest trends in AI-powered marketing tools. Identify at least 3 key trends and provide a brief explanation for each.",
    agent=researcher_agent,
    expected_output="A report outlining 3 key AI-powered marketing trends with explanations."
)

writing_task = Task(
    description="Based on the research report, write a blog post of at least 5 paragraphs highlighting the identified AI marketing trends and their potential impact on businesses.",
    agent=writer_agent,
    expected_output="A well-written blog post about AI-powered marketing trends."
)


In [53]:
# Create the Crew with Hierarchical Process
crew = Crew(
    agents=[manager_agent, researcher_agent, writer_agent],
    tasks=[research_task, writing_task],
    process=Process.hierarchical,
    manager_llm=openai_llm, # Use the same openai_llm for the manager
    verbose=True
)


2025-04-18 09:39:50,660 - 8618379328 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [54]:
print("Starting the hierarchical crew...")
result = crew.kickoff()
print("\n\n========= Crew Completed =========")
print(result)

Starting the hierarchical crew...
# Agent: Crew Manager
## Task: Conduct market research on the latest trends in AI-powered marketing tools. Identify at least 3 key trends and provide a brief explanation for each.
# Agent: Market Researcher
## Task: Conduct market research on the latest trends in AI-powered marketing tools. Identify at least 3 key trends with explanations.


# Agent: Market Researcher
## Final Answer: 
In the rapidly evolving landscape of marketing technology, AI-powered tools are at the forefront of innovation, driving significant shifts in strategies and operational efficiencies. Our comprehensive market research has identified three key trends that are shaping the future of AI in marketing. These trends not only highlight the current state of AI-powered marketing tools but also forecast the direction of future developments. Here’s a detailed analysis of each trend, supported by data and case studies to inform our strategic planning and decision-making process.

1. *

# Agent with Tools

In [55]:
# from crewai import Tool
from crewai_tools import SerperDevTool  # An in-built tool for web searching
from datetime import datetime


In [56]:

# Define a User-Defined Tool (same as before)
class CurrentYearTool():
    name = "current_year_tool"
    description = "Useful for getting the current year."

    def func(self, query: str = "") -> str:
        now = datetime.now()
        year = now.year
        # location_context = "in Bhopal, Madhya Pradesh, India"
        return f"The current year is {year}"

# Instantiate the user-defined tool
time_tool = CurrentYearTool()


In [ ]:
# Instantiate an In-built Tool from crewai_tools
search_tool = SerperDevTool()

os.environ["SERPER_API_KEY"] = ""


In [58]:
# Create an Agent with a mix of tools
research_agent = Agent(
    role="Research Analyst",
    goal="Conduct research on a given topic based on the given year. You need to find what was the state of art of that topic in the current year.",
    backstory="An expert researcher who can leverage web search and provide timely information.",
    llm=openai_llm,
    tools=[search_tool, time_tool],
    verbose=True
)


In [59]:
# Create a Task for the Agent
user_inquiry = "Find out the latest news about AI advancements in healthcare in the current year."
research_task = Task(
    description=f"Research the latest advancements in AI within the healthcare industry and provide a summary of the key findings.",
    agent=research_agent,
    expected_output="A summary of recent AI advancements in healthcare in the current year."
)

In [60]:
# Create a Crew with the Agent and Task
mixed_tools_crew = Crew(
    agents=[research_agent],
    tasks=[research_task],
    verbose=True
)

2025-04-18 09:41:27,881 - 8618379328 - __init__.py-__init__:537 - WARNING: Overriding of current TracerProvider is not allowed


In [61]:
# Run the Crew
print("Starting the crew with mixed tools...")
result = mixed_tools_crew.kickoff()
print("\n\n========= Crew Completed =========")
print(result)

Starting the crew with mixed tools...
# Agent: Research Analyst
## Task: Research the latest advancements in AI within the healthcare industry and provide a summary of the key findings.


# Agent: Research Analyst
## Thought: I need to find the latest advancements in AI within the healthcare industry for the current year to provide a comprehensive summary.
## Using tool: current_year_tool
## Tool Input: 
"{\"query\": \"current year\"}"
## Tool Output: 
The current year is 2025


# Agent: Research Analyst
## Thought: Thought: Now that I know the current year is 2025, I'll search for the latest advancements in AI within the healthcare industry for this year.
## Using tool: Search the internet with Serper
## Tool Input: 
"{\"search_query\": \"AI advancements in healthcare 2025\"}"
## Tool Output: 
{'searchParameters': {'q': 'AI advancements in healthcare 2025', 'type': 'search', 'num': 10, 'engine': 'google'}, 'organic': [{'title': 'AI in healthcare: What to expect in 2025', 'link': 'http